In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "8"

In [ ]:
from ultralytics import YOLO
import pandas as pd


In [ ]:
def eval_pose(model_path, data_yaml="../configs/data.yaml", split="val"):
    model = YOLO(model_path)
    results = model.val(
        data=data_yaml,
        split=split,
        verbose=False
    )
    
    metrics = {
        "model": model_path,
        "pose_mAP50": results.pose.map50,
        "pose_mAP50_95": results.pose.map,
        "pose_P": results.pose.mp,
        "pose_R": results.pose.mr,
        "speed_inference(ms)": results.speed["inference"]
    }
    return metrics


In [ ]:
m_teacher = eval_pose("../weights/teacher/yolom_kpt.pt")
m_teacher

In [ ]:
n_baseline = eval_pose("../weights/student/yolo11n-base.pt")
n_baseline

In [ ]:
import torch

def eval_pose_clean(
    clean_weight,
    model_yaml="../configs/yolo11n-pose.yaml",
    data_yaml="../configs/data.yaml"
):
    model = YOLO(model_yaml)
    state_dict = torch.load(clean_weight, map_location="cpu")
    model.model.load_state_dict(state_dict, strict=True)

    results = model.val(data=data_yaml, verbose=False)

    return {
        "pose_mAP50": results.pose.map50,
        "pose_mAP50_95": results.pose.map,
        "pose_P": results.pose.mp,
        "pose_R": results.pose.mr,
        "speed_inference(ms)": results.speed["inference"]
    }


In [ ]:
n_distilled = eval_pose_clean("../weights/export/yolo11n_pose21_clean.pt")
n_distilled

In [ ]:
n_distilled_onnx = eval_pose("../weights/export/yolo11n-pose.onnx")
n_distilled_onnx

In [ ]:
df = pd.DataFrame([
    m_teacher,
    n_baseline,
    n_distilled,
    # n_distilled_onnx
])

df


In [ ]:
from ultralytics import YOLO
import numpy as np

def eval_pose_ultralytics(model_path, data_yaml="../configs/data.yaml"):
    model = YOLO(model_path)

    results = model.val(
        data=data_yaml,
        split="val",
        imgsz=640,
        conf=0.25,
        device="cpu",
        verbose=False
    )

    return {
        "model": model_path,
        "pose_mAP50": results.pose.map50,
        "pose_mAP50_95": results.pose.map,
        "pose_P": results.pose.mp,
        "pose_R": results.pose.mr,
        "inference_ms": results.speed["inference"]
    }


In [ ]:
models = {
    "m_finetuned": "../weights/teacher/yolom_kpt.pt",
    "n_baseline": "../weights/student/yolo11n-base.pt",
    # "n_distilled": "../weights/export/yolo11n_pose21_clean.pt",
    "n_distilled_onnx": "../weights/export/yolo11n-pose.onnx",
}

results = [eval_pose_ultralytics(p) for p in models.values()]
results


In [ ]:
eval_pose_ultralytics("../weights/export/yolo11n-pose.onnx")